# Atera kidney: review candidate cells with Yearbook

Use a tissue-wide **Landscape** for context, then send **100 candidate cell IDs**
to a paginated **Yearbook**. Check portraits that deserve a closer look and retrieve
those IDs in Python for follow-up or annotation.

Here, coexpression of **EPCAM** and **PTPRC** is a simple screening proxy for possible
mixed-lineage signal, segmentation merges, or transcript assignment to nearby cells.
These are **review candidates, not doublet calls**. Coexpression or high counts alone
cannot distinguish technical artifacts from biology.

The data are the public [10x Genomics Atera WTA FFPE human non-diseased kidney
dataset](https://www.10xgenomics.com/datasets/atera-wta-ffpe-human-nd-kidney).
We use the same [prepared DegaFiles](https://github.com/broadinstitute/Celldega_Atera_v1_WTA_Kidney_non_diseased_FFPE)
as the [Atera kidney tutorial](https://broadinstitute.github.io/celldega/examples/tutorial_notebooks/Atera_Kidney_Landscape-Clustergram/).
The default analysis downloads only cell IDs/positions and two sparse gene vectors
(about **8.5 MB**); the viewers stream additional images and spatial tiles as needed.

Use the **current Celldega checkout** for the new portrait checkboxes and Python
selection retrieval: install it with `pip install -e .` and build its front end
with `npm run build`. Start a fresh kernel; the first cell sets
`CELLDEGA_LOCAL_ESM=1` before importing Celldega. Run with a live kernel to
export checked IDs.
The rendered documentation also supports browsing and checking portraits, but
cannot execute the Python retrieval cells.

In [1]:
import os
from pathlib import Path
from urllib.request import urlretrieve
import json

# Use the checkout's matching front end for these new Yearbook controls.
os.environ["CELLDEGA_LOCAL_ESM"] = "1"

import anndata as ad
import numpy as np
import pandas as pd
from scipy import sparse
import celldega as dega

assert hasattr(dega.viz.Yearbook, "get_selected_cells"), "Install the current Celldega checkout."


In [2]:
SAMPLE = "Atera_v1_WTA_Kidney_non_diseased_FFPE"
DATA_REVISION = "3a46ab4db67821eef3b8ff8d1c3704fb7b3bb48f"
BASE_URL = f"https://raw.githubusercontent.com/broadinstitute/Celldega_{SAMPLE}/{DATA_REVISION}"
ANNDATA_URL = f"https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/resolve/main/{SAMPLE}.h5ad"

cwd = Path.cwd().resolve()
repo = next((p for p in [cwd, *cwd.parents] if (p / "pyproject.toml").is_file()), None)
cache = repo / "data/atera_yearbook_review" if repo else cwd / "data/atera_yearbook_review"
cache.mkdir(parents=True, exist_ok=True)

for relative in ["landscape_parameters.json", "micron_to_image_transform.csv"]:
    path = cache / relative
    if not path.exists():
        urlretrieve(f"{BASE_URL}/{relative}", path)
parameters = json.loads((cache / "landscape_parameters.json").read_text())
transform = np.loadtxt(cache / "micron_to_image_transform.csv")

## 1. Inspect the tissue context

This Landscape shows the whole section with its original default
clusters. Use its gene controls to examine EPCAM and PTPRC in tissue context;
the Yearbook below will focus on a small set of cells.

In [3]:
landscape = dega.viz.Landscape(
    technology="Xenium", base_url=BASE_URL, transform=transform,
    dataset_name="Atera kidney — tissue context", width=0, height=600,
    max_tiles_to_view=25,
)
landscape.layout.width = "100%"
landscape

## 2. Combine marker thresholds and rank candidates with `dega.select`

Build a small AnnData containing just EPCAM and PTPRC counts for all **399,120**
cells. This pinned DegaFiles export uses numeric row positions in its sparse gene
files; the observation index below preserves the original cell IDs from
`cell_metadata.parquet`.

`Selector.gene(...)` builds an expression predicate, `&` combines predicates,
and `selector.samplers.rank(...)` chooses a bounded, ordered review set. These
thresholds are exploratory settings that you can change. See the
[Select API](https://broadinstitute.github.io/celldega/python/select/api/) for more query and sampling options.

In [4]:
MARKERS = ["EPCAM", "PTPRC"]
for relative in ["cell_metadata.parquet", *[f"cbg/{gene}.parquet" for gene in MARKERS]]:
    path = cache / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        urlretrieve(f"{BASE_URL}/{relative}", path)

cell_ids = pd.Index(
    pd.read_parquet(cache / "cell_metadata.parquet", columns=["name"])["name"].astype(str),
    name="cell_id",
)
assert cell_ids.is_unique
assert parameters["use_int_index"], "This example expects the pinned indexed DegaFiles."

columns = []
for gene in MARKERS:
    frame = pd.read_parquet(cache / "cbg" / f"{gene}.parquet")
    rows = frame.index.to_numpy(dtype=np.int64)
    assert ((rows >= 0) & (rows < len(cell_ids))).all()
    columns.append(sparse.csc_matrix(
        (frame[gene].to_numpy(), (rows, np.zeros(len(rows), dtype=np.int64))),
        shape=(len(cell_ids), 1),
    ))

cells = ad.AnnData(
    X=sparse.hstack(columns, format="csr"),
    obs=pd.DataFrame(index=cell_ids),
    var=pd.DataFrame(index=MARKERS),
)
print(f"{cells.n_obs:,} cells; {cells.n_vars} marker genes; original cell IDs retained")

399,120 cells; 2 marker genes; original cell IDs retained


In [5]:
selector = dega.select.Selector(cells)
mixed_markers = (selector.gene("EPCAM") >= 2) & (selector.gene("PTPRC") >= 2)
mixed_review = selector.select(
    query=mixed_markers,
    sampler=selector.samplers.rank(attr=selector.gene("PTPRC"), n=100, by="high"),
)
# To sample the matching candidates instead, use:
# sampler=selector.samplers.random(n=100, seed=0)

print(f"{mixed_review.candidate_count:,} candidates; {len(mixed_review)} cells for review")
mixed_review.to_dataframe().head()

198 candidates; 100 cells for review


,id,rank,score
0,bckobfja-1,0,5.0
1,lfpdnjai-1,1,5.0
2,adfpjbkn-1,2,4.0
3,cbofpihm-1,3,4.0
4,fedodihm-1,4,4.0


## 3. Review portraits across pages

Pass the ordered list of original cell IDs to Yearbook. With **15 portraits per
page**, 100 cells span seven
pages. Use the checkbox on each portrait to flag a cell; checked IDs remain selected
when you change pages. Inspect cell boundaries, nearby nuclei, and marker transcripts
before deciding which cells warrant follow-up.

The checked IDs are separate from `yearbook.cells`, which is the complete queue of
candidate cells shown by the viewer. You can also pass `selection=review_selection`
instead of `cells=candidate_ids` to store the query/ranking provenance in the widget;
the export below records that provenance either way.

In [6]:
review_selection = mixed_review
candidate_ids = review_selection.names()
yearbook = dega.viz.Yearbook(
    name="kidney_candidate_review", base_url=BASE_URL, cells=candidate_ids,
    rows=3, cols=5, portrait_size_um=50,
    dataset_name="Atera kidney — candidate review", width=0, height=850,
)
yearbook.layout.width = "100%"
yearbook

## 4. Retrieve checked cells and export review flags

**Run this cell after checking portraits in a live notebook.** `selected_cells`
contains checked original IDs; `get_selected_cells(cells)` returns a copied AnnData
subset. Unchecked candidates have not automatically passed QC.

The CSV records the full review queue plus the manual flag. The JSON records both
the query/ranking provenance and checked IDs. The same ID-based annotation can be
applied to your full expression AnnData without changing its counts.

In [ ]:
checked_ids = yearbook.get_selected_cells()  # same IDs as yearbook.selected_cells
flagged_cells = yearbook.get_selected_cells(cells)
cells.obs["yearbook_review_flag"] = cells.obs_names.isin(checked_ids)

review_flags = pd.DataFrame({"cell_id": candidate_ids})
review_flags["yearbook_review_flag"] = review_flags["cell_id"].isin(checked_ids)
export_dir = cache / "review_exports"
export_dir.mkdir(exist_ok=True)
review_flags.to_csv(export_dir / "candidate_review_flags.csv", index=False)
(export_dir / "candidate_review.json").write_text(json.dumps({
    "selection": review_selection.to_json(),
    "checked_ids": checked_ids,
}, indent=2))

# For an in-memory full expression AnnData with these same original IDs:
# adata.obs["yearbook_review_flag"] = adata.obs_names.isin(checked_ids)
print(f"{len(checked_ids)} checked cells; review files saved under {export_dir}")

## Optional: sample high-count cells using observation metadata

A second `dega.select` example combines an `obs` threshold with seeded random
sampling. The [annotated counts H5AD](https://huggingface.co/datasets/broadinstitute/Celldega_Supporting_Data/blob/main/Atera_v1_WTA_Kidney_non_diseased_FFPE.h5ad)
contains `total_counts` in `obs`. If you already downloaded it for the full kidney
tutorial, the code below reads only metadata in backed mode, not the count matrix.
It deliberately does **not** download that 1.31 GiB file just for this example.

Cells in the highest 1% of total counts are another review queue, not a doublet
classification. To inspect this alternative, replace `review_selection = mixed_review`
with `review_selection = high_count_review` in the Yearbook cell and rerun it.

In [7]:
source_path = (
    repo / "notebooks/data/atera_data" / SAMPLE / f"{SAMPLE}.h5ad"
    if repo else cwd / "data" / f"{SAMPLE}.h5ad"
)
high_count_review = None
if source_path.exists():
    source = ad.read_h5ad(source_path, backed="r")
    qc_cells = ad.AnnData(obs=source.obs[["total_counts"]].copy())
    source.file.close()
    qc_selector = dega.select.Selector(qc_cells)
    count_cutoff = qc_cells.obs["total_counts"].quantile(0.99)
    high_count_review = qc_selector.select(
        query=qc_selector.attr("total_counts") >= count_cutoff,
        sampler=qc_selector.samplers.random(n=100, seed=0),
    )
    print(f"total_counts >= {count_cutoff:g}: {high_count_review.candidate_count:,} candidates; "
          f"{len(high_count_review)} sampled for review")
else:
    print("Optional high-count example skipped: no cached H5AD. The marker-based review above is ready.")

total_counts >= 9225: 3,993 candidates; 100 sampled for review
